In [6]:

#la fusion de la table dépasse plus d'un million de lignes donc on ne peut pas fusionner les données dans Microsoft Excel, il faut donc utiliser Python pour fusionner les fichiers Excel
# Feuille YEAR 2009_2010: 525 462 lignes
# Feuille YEAR 2010_2011: 541 911 lignes
! pip install pandas


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd

In [8]:
# Fichier Excel
fichier = "online_retail.xlsx"

# Lire les deux feuilles
df1 = pd.read_excel(fichier, sheet_name="Year 2009-2010")
df2 = pd.read_excel(fichier, sheet_name="Year 2010-2011")

# Fusionner les deux tables
df = pd.concat([df1, df2], ignore_index=True)

# Enregistrer en CSV
df.to_csv("online_retail_fusionne.csv", index=False, encoding="utf-8-sig")

print(f"Fusion terminée : {len(df)} lignes")

Fusion terminée : 1067371 lignes


In [9]:
#maintenant on va requêter la table fusionnée avec DuckDB ( SQL) pour faire des analyses sur les données

In [21]:
!pip install duckdb jupysql


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
%load_ext sql

%sql duckdb:///online_retail.duckdb

Connecting to 'duckdb:///online_retail.duckdb'

In [23]:
#créer une table dans DuckDB à partir du fichier CSV fusionné

In [13]:
%%sql
CREATE OR REPLACE TABLE online_retail AS
SELECT *
FROM read_csv_auto('online_retail_fusionne.csv');

Running query in 'duckdb:///online_retail.duckdb'

Count


In [15]:
%%sql
SELECT * 
FROM online_retail
LIMIT 5;

Running query in 'duckdb:///online_retail.duckdb'

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [17]:
%%sql
SELECT 
COUNT(DISTINCT "Customer ID") AS nbr_valeur_nulle_customer
FROM online_retail
WHERE "Customer ID" IS NULL;

Running query in 'duckdb:///online_retail.duckdb'

nbr_valeur_nulle_customer
0


In [ ]:
#TOTAL QUANTITY

In [34]:
%%sql
SELECT
SUM("Quantity") AS QTT_TOTAL
FROM online_retail;

Running query in 'duckdb:///online_retail.duckdb'

QTT_TOTAL
10608492


In [ ]:
#TOTAL PRICE

In [39]:
%%sql
SELECT SUM(ROUND("Price")) AS PRICE_TOTAL FROM online_retail;

Running query in 'duckdb:///online_retail.duckdb'

PRICE_TOTAL
4979243.0


In [ ]:
#TOTAL_CHIFFRE_D'AFFAIRE

In [44]:
%%sql

SELECT
    YEAR("InvoiceDate") AS ANNEE,
    SUM(ROUND("Price" * "Quantity")) AS CHIFFRE_D_AFFAIRE_TOTAL
FROM online_retail
WHERE YEAR("InvoiceDate") IN (2009, 2010, 2011)
GROUP BY YEAR("InvoiceDate")
ORDER BY ANNEE;

Running query in 'duckdb:///online_retail.duckdb'

ANNEE,CHIFFRE_D_AFFAIRE_TOTAL
2009,802637.0
2010,9523755.0
2011,9023112.0
